# 🌦️ NDVI estación seca vs. estación lluviosa
## Google Earth Engine + Google Colab

[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/GrettelVargasAz/gee-cloud-webinar/blob/main/scripts/Colab_02_NDVI_Seca_vs_Lluviosa.ipynb)

**Caso de estudio:** departamento de Zacapa, Guatemala (Corredor Seco Centroamericano)

---

### ¿Por qué comparar dos épocas?
En el Corredor Seco, un NDVI bajo en marzo **no significa necesariamente degradación**: buena parte del bosque seco es caducifolio y pierde las hojas en la estación seca. Comparar ambas épocas permite separar:

- vegetación que se mantiene verde todo el año (bosque de altura, ribera, riego),
- vegetación que responde a la lluvia (bosque caducifolio, pastos, cultivos de temporal),
- áreas con poca o ninguna vegetación en ambas épocas.

### ¿Qué vamos a obtener?
1. Un **NDVI de estación seca** y un **NDVI de estación lluviosa**, cada uno como compuesto multianual sin nubes.
2. La **diferencia dNDVI = lluviosa − seca** (respuesta estacional).
3. Una **clasificación exploratoria** en cuatro tipos de respuesta, con su superficie en hectáreas.
4. Estadísticas, histograma y mapas comparativos.
5. Cuatro **GeoTIFF** descargados en un `.zip` (o guardados en Google Drive), listos para ArcGIS Pro o QGIS.

### Flujo de trabajo

```
                     ┌─► meses secos ───► máscara nubes ─► mediana ─► NDVI seca ─────┐
Sentinel-2 ─► área ──┤                                                               ├─► dNDVI ─► clases
                     └─► meses lluviosos ► máscara nubes ─► mediana ─► NDVI lluviosa ┘
                                                                                     │
          ArcGIS Pro / QGIS ◄── descarga o Drive ◄── GeoTIFF ◄── estadísticas + mapas ◄─┘
```

> 💡 **Este notebook corre en Google Colab**: no hay que instalar nada en su computadora. Solo necesita una cuenta de Google y un proyecto registrado en Earth Engine. Al final, el GeoTIFF se descarga a su computadora o se guarda en Google Drive, listo para abrir en **ArcGIS Pro** o **QGIS**.
>
> Use **Archivo ▸ Guardar una copia en Drive** para guardar sus cambios.

## 1. Importar librerías

Colab ya trae `earthengine-api` y `matplotlib`. Si `geemap` no está disponible, la celda lo instala.

In [ ]:
import subprocess, sys

try:
    import geemap
except ImportError:
    print("Instalando geemap (solo la primera vez, ~1 min)...")
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'geemap'], check=True)
    import geemap

import os
import ee
from google.colab import files
import matplotlib.pyplot as plt
import zipfile

print("✓ Librerías cargadas")
print(f"  earthengine-api : {ee.__version__}")
print(f"  geemap          : {geemap.__version__}")

## 2. Parámetros

👉 **Única celda que necesita modificar.**

| Parámetro | Descripción |
|---|---|
| `PROYECTO_GEE` | ID de su proyecto de Google Cloud registrado en Earth Engine |
| `PAIS`, `DEPTO` | Nombres tal como aparecen en FAO GAUL (nivel 1) |
| `ANIO_INICIO`, `ANIO_FIN` | Años que se combinan (ambos incluidos). Varios años dan un valor "típico" y menos afectado por un año atípico |
| `MESES_SECA` | Meses de la estación seca (inicio, fin). Feb–abr es el pico seco: el bosque caducifolio ya perdió las hojas |
| `MESES_LLUVIA` | Meses de la estación lluviosa. Sep–oct: vegetación en su máximo verdor, después de la canícula |
| `MAX_NUBES_*` | % máximo de nubosidad por escena. En la época lluviosa hay que ser más tolerante; la máscara SCL limpia el resto |
| `UMBRAL_*` | Umbrales de la clasificación exploratoria (sección 9) |

In [ ]:
# --- Conexión ---------------------------------------------------------------
PROYECTO_GEE = 'su-proyecto-gee'        # ← reemplazar por el ID de su proyecto

# --- Área de estudio --------------------------------------------------------
PAIS  = 'Guatemala'
DEPTO = 'Zacapa'

# --- Épocas (meses incluidos) y años combinados -----------------------------
ANIO_INICIO  = 2023
ANIO_FIN     = 2025          # incluido
MESES_SECA   = (2, 4)        # febrero a abril
MESES_LLUVIA = (9, 10)       # septiembre a octubre

# --- Calidad ----------------------------------------------------------------
MAX_NUBES_SECA   = 20
MAX_NUBES_LLUVIA = 50

# --- Clasificación exploratoria ---------------------------------------------
UMBRAL_VERDE      = 0.5      # NDVI que se considera vegetación vigorosa
UMBRAL_RESPUESTA  = 0.2      # dNDVI mínimo para considerar respuesta estacional
UMBRAL_SIN_VEG    = 0.2      # NDVI lluvioso por debajo del cual no hay vegetación

# --- Salida -----------------------------------------------------------------
ESCALA_M   = 30
CRS_SALIDA = 'EPSG:32616'    # WGS 84 / UTM 16N
NODATA     = -9999

MESES = ['ene','feb','mar','abr','may','jun','jul','ago','sep','oct','nov','dic']
etiqueta_seca   = f"{MESES[MESES_SECA[0]-1]}–{MESES[MESES_SECA[1]-1]}"
etiqueta_lluvia = f"{MESES[MESES_LLUVIA[0]-1]}–{MESES[MESES_LLUVIA[1]-1]}"

print(f"Área            : {DEPTO}, {PAIS}")
print(f"Años            : {ANIO_INICIO}–{ANIO_FIN}")
print(f"Estación seca   : {etiqueta_seca}")
print(f"Estación lluviosa: {etiqueta_lluvia}")

## 3. Conectar con Google Earth Engine

La primera vez, Colab pide autorizar el acceso con su cuenta de Google. Acepte los permisos y la conexión queda lista para esta sesión.

In [ ]:
if PROYECTO_GEE == 'su-proyecto-gee':
    raise ValueError("Defina PROYECTO_GEE en la sección 2 con el ID de su proyecto de Earth Engine.")

try:
    ee.Initialize(project=PROYECTO_GEE)
    print(f"✓ Conectado a Google Earth Engine (proyecto: {PROYECTO_GEE})")
except Exception:
    print("Autenticando Google Earth Engine...")
    ee.Authenticate()
    ee.Initialize(project=PROYECTO_GEE)
    print(f"✓ Autenticación completada (proyecto: {PROYECTO_GEE})")

## 4. Área de estudio

In [ ]:
limites = ee.FeatureCollection('FAO/GAUL/2015/level1')
depto = limites.filter(ee.Filter.And(
    ee.Filter.eq('ADM0_NAME', PAIS),
    ee.Filter.eq('ADM1_NAME', DEPTO)
))

if depto.size().getInfo() == 0:
    nombres = limites.filter(ee.Filter.eq('ADM0_NAME', PAIS)).aggregate_array('ADM1_NAME').sort().getInfo()
    raise ValueError(f"No se encontró '{DEPTO}'. Nombres disponibles en {PAIS}:\n{nombres}")

geom = depto.geometry()
area_km2 = geom.area(maxError=100).divide(1e6).getInfo()
lon_c, lat_c = geom.centroid(maxError=100).coordinates().getInfo()

print(f"✓ {DEPTO}: {area_km2:,.0f} km²  |  centro: lat {lat_c:.3f}, lon {lon_c:.3f}")

## 5. Compuestos por época

Una sola función construye el compuesto de cualquier época:

1. Filtra Sentinel-2 L2A armonizado por área y por años.
2. Con `calendarRange` conserva **solo los meses de la época** en cada año.
3. Descarta escenas muy nubladas y aplica la máscara de nubes **SCL** (clases 3, 8, 9 y 10).
4. Combina todo con la **mediana**.

También contamos las **observaciones válidas por píxel**: en la época lluviosa algunas zonas pueden quedar con pocas observaciones libres de nubes.

In [ ]:
def enmascarar_nubes(img):
    scl = img.select('SCL')
    nubes = scl.eq(3).Or(scl.eq(8)).Or(scl.eq(9)).Or(scl.eq(10))
    return img.updateMask(nubes.Not())


def coleccion_epoca(meses, max_nubes):
    return (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterBounds(geom)
        .filterDate(f'{ANIO_INICIO}-01-01', f'{ANIO_FIN + 1}-01-01')
        .filter(ee.Filter.calendarRange(meses[0], meses[1], 'month'))
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', max_nubes))
        .map(enmascarar_nubes)
    )


col_seca   = coleccion_epoca(MESES_SECA, MAX_NUBES_SECA)
col_lluvia = coleccion_epoca(MESES_LLUVIA, MAX_NUBES_LLUVIA)

n_seca, n_lluvia = col_seca.size().getInfo(), col_lluvia.size().getInfo()
if n_seca == 0 or n_lluvia == 0:
    raise ValueError("Una de las épocas no tiene imágenes. Amplíe los años o aumente MAX_NUBES_*.")

s2_seca   = col_seca.median().clip(geom)
s2_lluvia = col_lluvia.median().clip(geom)

# Observaciones válidas por píxel (calidad del compuesto)
obs_seca   = col_seca.select('B8').count().rename('obs')
obs_lluvia = col_lluvia.select('B8').count().rename('obs')

def obs_resumen(img):
    r = img.reduceRegion(ee.Reducer.min().combine(ee.Reducer.median(), sharedInputs=True),
                         geom, ESCALA_M * 4, maxPixels=1e10, bestEffort=True).getInfo()
    return r['obs_min'], r['obs_median']

print(f"Escenas estación seca    ({etiqueta_seca})  : {n_seca:4d}  | obs. válidas por píxel (mín / mediana): {obs_resumen(obs_seca)}")
print(f"Escenas estación lluviosa ({etiqueta_lluvia}): {n_lluvia:4d}  | obs. válidas por píxel (mín / mediana): {obs_resumen(obs_lluvia)}")

## 6. NDVI de cada época y diferencia

$$dNDVI = NDVI_{lluviosa} - NDVI_{seca}$$

- **dNDVI alto:** la vegetación "reverdece" con la lluvia (caducifolio, pastos, cultivos de temporal).
- **dNDVI cercano a 0:** cobertura estable, ya sea siempreverde o sin vegetación.
- **dNDVI negativo:** más verde en la época seca que en la lluviosa (riego en verano, cosecha o quema antes de la toma lluviosa, o nubes residuales).

In [ ]:
ndvi_seca   = s2_seca.normalizedDifference(['B8', 'B4']).rename('NDVI_seca')
ndvi_lluvia = s2_lluvia.normalizedDifference(['B8', 'B4']).rename('NDVI_lluviosa')
dndvi       = ndvi_lluvia.subtract(ndvi_seca).rename('dNDVI')

print("✓ NDVI seca, NDVI lluviosa y dNDVI calculados")

## 7. Estadísticas comparativas

In [ ]:
pila = ndvi_seca.addBands(ndvi_lluvia).addBands(dndvi)

reductor = (
    ee.Reducer.mean()
    .combine(ee.Reducer.percentile([10, 50, 90]), sharedInputs=True)
)
stats = pila.reduceRegion(reductor, geom, ESCALA_M, maxPixels=1e10, bestEffort=True).getInfo()

print(f"{'':16s}{'Media':>8s}{'P10':>8s}{'P50':>8s}{'P90':>8s}")
for banda, etiqueta in [('NDVI_seca', f'Seca ({etiqueta_seca})'),
                        ('NDVI_lluviosa', f'Lluviosa ({etiqueta_lluvia})'),
                        ('dNDVI', 'dNDVI')]:
    print(f"{etiqueta:16s}" + "".join(f"{stats[f'{banda}_{k}']:8.3f}" for k in ['mean', 'p10', 'p50', 'p90']))

### Histograma

Se toma una muestra aleatoria de píxeles para comparar la distribución de ambas épocas. Normalmente la curva seca está desplazada hacia la izquierda.

In [ ]:
muestra = pila.sample(region=geom, scale=ESCALA_M, numPixels=5000, seed=42, geometries=False)
valores = muestra.reduceColumns(ee.Reducer.toList(3), ['NDVI_seca', 'NDVI_lluviosa', 'dNDVI']).get('list').getInfo()
v_seca, v_lluvia, v_dif = zip(*valores)

fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
ejes[0].hist(v_seca,   bins=50, range=(-0.2, 1), alpha=0.6, color='#d8863b', label=f'Seca ({etiqueta_seca})')
ejes[0].hist(v_lluvia, bins=50, range=(-0.2, 1), alpha=0.6, color='#2e7d4f', label=f'Lluviosa ({etiqueta_lluvia})')
ejes[0].set(title=f'Distribución del NDVI — {DEPTO}', xlabel='NDVI', ylabel='Píxeles')
ejes[0].legend()

ejes[1].hist(v_dif, bins=50, range=(-0.4, 0.8), color='#2f5d8a')
ejes[1].axvline(UMBRAL_RESPUESTA, color='k', ls='--', lw=1, label=f'Umbral respuesta ({UMBRAL_RESPUESTA})')
ejes[1].set(title='dNDVI (lluviosa − seca)', xlabel='dNDVI', ylabel='Píxeles')
ejes[1].legend()

plt.tight_layout()
plt.show()
print(f"Muestra: {len(v_seca):,} píxeles")

## 8. Comparación visual: mapa dividido

Arrastre la barra central para comparar ambas épocas en el mismo lugar.

In [ ]:
paleta_ndvi = ['#a50026', '#f46d43', '#fee08b', '#d9ef8b', '#66bd63', '#006837']
vis_ndvi = {'min': 0, 'max': 0.8, 'palette': paleta_ndvi}

capa_izq = geemap.ee_tile_layer(ndvi_seca,   vis_ndvi, f'NDVI seca ({etiqueta_seca})')
capa_der = geemap.ee_tile_layer(ndvi_lluvia, vis_ndvi, f'NDVI lluviosa ({etiqueta_lluvia})')

MapaSplit = geemap.Map(center=[lat_c, lon_c], zoom=9, toolbar_ctrl=False)
MapaSplit.split_map(left_layer=capa_izq, right_layer=capa_der)
MapaSplit.add_colorbar(vis_ndvi, label='NDVI')
MapaSplit

## 9. Clasificación exploratoria por respuesta estacional

Reglas simples, aplicadas en este orden (los umbrales se ajustan en la sección 2):

| Clase | Nombre | Regla | Coberturas típicas |
|---|---|---|---|
| 1 | Verde todo el año | NDVI seca ≥ `UMBRAL_VERDE` | Bosque de altura, ribera, cultivos bajo riego |
| 2 | Respuesta estacional fuerte | NDVI lluviosa ≥ `UMBRAL_VERDE` y dNDVI ≥ `UMBRAL_RESPUESTA` | Bosque seco caducifolio, cultivos de temporal |
| 3 | Vegetación escasa / estacional débil | NDVI lluviosa ≥ `UMBRAL_SIN_VEG` (resto) | Matorral, pastos, áreas degradadas |
| 4 | Sin vegetación | NDVI lluviosa < `UMBRAL_SIN_VEG` | Suelo desnudo, urbano, roca, agua |

> ⚠️ Es una **estratificación exploratoria**, útil para orientar el análisis y el muestreo de campo. **No sustituye** una clasificación de cobertura validada.

In [ ]:
clases = (
    ee.Image(3)
    .where(ndvi_lluvia.lt(UMBRAL_SIN_VEG), 4)
    .where(ndvi_lluvia.gte(UMBRAL_VERDE).And(dndvi.gte(UMBRAL_RESPUESTA)), 2)
    .where(ndvi_seca.gte(UMBRAL_VERDE), 1)
    .updateMask(dndvi.mask())
    .clip(geom)
    .rename('clase')
    .toByte()
)

NOMBRES_CLASE = {
    1: 'Verde todo el año',
    2: 'Respuesta estacional fuerte',
    3: 'Vegetación escasa / estacional débil',
    4: 'Sin vegetación',
}
COLORES_CLASE = {1: '#1b5e20', 2: '#8bc34a', 3: '#e6c36a', 4: '#bdbdbd'}

# Superficie por clase (ha)
areas = (
    ee.Image.pixelArea().divide(1e4).addBands(clases)
    .reduceRegion(
        reducer=ee.Reducer.sum().group(groupField=1, groupName='clase'),
        geometry=geom, scale=ESCALA_M, maxPixels=1e10, bestEffort=True
    ).get('groups').getInfo()
)
total_ha = sum(g['sum'] for g in areas)

print(f"{'Clase':40s}{'Hectáreas':>12s}{'%':>8s}")
for g in sorted(areas, key=lambda g: g['clase']):
    c = int(g['clase'])
    print(f"{c}. {NOMBRES_CLASE[c]:37s}{g['sum']:12,.0f}{100 * g['sum'] / total_ha:7.1f}%")

## 10. Mapa de diferencia y clases

In [ ]:
Map = geemap.Map(center=[lat_c, lon_c], zoom=9, toolbar_ctrl=False, layer_ctrl=True)

Map.addLayer(s2_lluvia, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, 'Color verdadero lluviosa', False)
Map.addLayer(s2_seca,   {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, 'Color verdadero seca', False)

vis_dif = {'min': -0.2, 'max': 0.5, 'palette': ['#b2182b', '#f7f7f7', '#a6d96a', '#1a9850']}
Map.addLayer(dndvi, vis_dif, 'dNDVI (lluviosa − seca)')

Map.addLayer(clases, {'min': 1, 'max': 4, 'palette': [COLORES_CLASE[i] for i in range(1, 5)]},
             'Respuesta estacional', False)

Map.addLayer(depto.style(color='000000', fillColor='00000000', width=2), {}, f'Límite de {DEPTO}')
Map.add_colorbar(vis_dif, label='dNDVI')
Map.add_legend(title='Respuesta estacional',
               legend_dict={NOMBRES_CLASE[i]: COLORES_CLASE[i].lstrip('#') for i in range(1, 5)})
Map

## 11. Exportar los cuatro GeoTIFF

| Archivo | Contenido | Tipo |
|---|---|---|
| `..._NDVI_seca.tif` | NDVI estación seca | Float32 |
| `..._NDVI_lluviosa.tif` | NDVI estación lluviosa | Float32 |
| `..._dNDVI.tif` | Diferencia lluviosa − seca | Float32 |
| `..._clases.tif` | Clasificación exploratoria (1–4) | Byte |
| `clases.clr` | Colores de las clases, para ArcGIS Pro | Texto |
| `clases_leyenda.csv` | Valor, nombre y color de cada clase | Texto |

**Opción A** (siguiente celda): los genera en Colab y descarga todo en un `.zip`.
**Opción B**: los envía a Google Drive ▸ `GEE_exports`; úsela si la opción A da error de tamaño.

In [ ]:
# --- Opción A: descarga directa en un .zip ------------------------------------

def declarar_nodata(ruta, valor):
    """Marca el valor como NoData dentro del GeoTIFF (si rasterio está disponible)."""
    try:
        import rasterio
        with rasterio.open(ruta, 'r+') as src:
            src.nodata = valor
        return True
    except Exception:
        return False


prefijo = f"{DEPTO.replace(' ', '_')}_{ANIO_INICIO}_{ANIO_FIN}"
carpeta = f"/content/{prefijo}"
os.makedirs(carpeta, exist_ok=True)

productos = {
    'NDVI_seca':     (ndvi_seca.toFloat(),   NODATA),
    'NDVI_lluviosa': (ndvi_lluvia.toFloat(), NODATA),
    'dNDVI':         (dndvi.toFloat(),       NODATA),
    'clases':        (clases,                0),
}

for clave, (imagen, nodata) in productos.items():
    ruta = os.path.join(carpeta, f"{prefijo}_{clave}.tif")
    print(f"Exportando {os.path.basename(ruta)} ...")
    geemap.ee_export_image(imagen, filename=ruta, scale=ESCALA_M, crs=CRS_SALIDA,
                           region=geom, unmask_value=nodata, file_per_band=False, verbose=False)
    if not os.path.exists(ruta):
        raise RuntimeError(f"No se generó {ruta}. Use la opción B (Google Drive).")
    declarar_nodata(ruta, nodata)

# Colores (.clr para ArcGIS Pro) y leyenda (.csv)
with open(os.path.join(carpeta, "clases.clr"), "w") as f, \
     open(os.path.join(carpeta, "clases_leyenda.csv"), "w", encoding="utf-8") as g:
    g.write("valor,clase,color\n")
    for c, hexcol in COLORES_CLASE.items():
        r, gr, b = (int(hexcol[i:i + 2], 16) for i in (1, 3, 5))
        f.write(f"{c} {r} {gr} {b}\n")
        g.write(f"{c},{NOMBRES_CLASE[c]},{hexcol}\n")

ruta_zip = f"/content/{prefijo}_NDVI_estacional.zip"
with zipfile.ZipFile(ruta_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for archivo in sorted(os.listdir(carpeta)):
        z.write(os.path.join(carpeta, archivo), archivo)

print(f"\n✓ {os.path.basename(ruta_zip)} ({os.path.getsize(ruta_zip) / 1e6:.1f} MB). Descargando...")
files.download(ruta_zip)

In [ ]:
# --- Opción B: exportar a Google Drive ----------------------------------------
prefijo = f"{DEPTO.replace(' ', '_')}_{ANIO_INICIO}_{ANIO_FIN}"
imagenes = {
    'NDVI_seca':     ndvi_seca.toFloat().unmask(NODATA),
    'NDVI_lluviosa': ndvi_lluvia.toFloat().unmask(NODATA),
    'dNDVI':         dndvi.toFloat().unmask(NODATA),
    'clases':        clases.unmask(0),
}

tareas = []
for clave, imagen in imagenes.items():
    nombre = f"{prefijo}_{clave}"
    tarea = ee.batch.Export.image.toDrive(
        image=imagen, description=nombre, folder='GEE_exports', fileNamePrefix=nombre,
        region=geom, scale=ESCALA_M, crs=CRS_SALIDA, maxPixels=1e10
    )
    tarea.start()
    tareas.append((nombre, tarea))
    print(f"✓ Enviada: {nombre}")

print("\nLos archivos aparecerán en Google Drive ▸ GEE_exports. Ejecute la siguiente celda para ver el avance.")

In [ ]:
# Estado de las exportaciones a Drive (ejecute de nuevo para actualizar)
for nombre, tarea in tareas:
    estado = tarea.status()
    print(f"{nombre:45s} {estado['state']}")

## 12. Abrir los resultados en ArcGIS Pro o QGIS

Descomprima el `.zip` y agregue los `.tif` al mapa.

**ArcGIS Pro**
- **NDVI seca, NDVI lluviosa, dNDVI:** *Symbology ▸ Stretch* con la rampa **Red-Yellow-Green**. Si los bordes se ven de un color uniforme, declare `-9999` como NoData con **Set Raster Properties**.
- **Clases:** *Symbology ▸ Unique Values*. Para usar los colores del notebook, aplique `clases.clr` con la herramienta **Add Colormap**.
- **Comparar épocas:** seleccione la capa de arriba y use **Raster Layer ▸ Compare ▸ Swipe**.

**QGIS**
- **NDVI:** *Propiedades ▸ Simbología ▸ Pseudocolor monobanda*, rampa **RdYlGn**.
- **Clases:** *Valores paletizados/únicos*; use `clases_leyenda.csv` como referencia de nombres y colores.
- **Comparar épocas:** complemento **MapSwipe Tool**.

## ✅ Cómo leer los resultados

| Lo que ve | Interpretación probable |
|---|---|
| NDVI alto en ambas épocas, dNDVI ≈ 0 | Cobertura siempreverde: bosque de altura, ribera, riego |
| NDVI bajo en seca, alto en lluviosa (dNDVI alto) | Vegetación estacional sana: bosque seco caducifolio, pastos, cultivos de temporal |
| NDVI bajo en ambas, dNDVI bajo | Suelo desnudo, zonas urbanas, áreas degradadas |
| dNDVI negativo | Riego en verano, cosecha o quema antes de la toma lluviosa, o nubes residuales: revisar |

**Para monitoreo de restauración:** compare siempre años distintos con **la misma ventana estacional**. Un sitio restaurado debería mostrar, con los años, un aumento del NDVI lluvioso y, en especies siempreverdes, también del NDVI seco.

---

## 🧪 Retos

1. **Cambiar las ventanas:** pruebe `MESES_SECA = (3, 4)` o `MESES_LLUVIA = (8, 10)`. ¿Cambian mucho las clases?
2. **Año contra año:** fije `ANIO_INICIO = ANIO_FIN = 2023` y repita con 2025. ¿Dónde aumentó el NDVI lluvioso?
3. **Ajustar umbrales:** revise el histograma de dNDVI y modifique `UMBRAL_RESPUESTA`.
4. **Validar:** superponga sus parcelas de campo o polígonos de restauración y calcule el NDVI medio por polígono con *Zonal Statistics as Table*.